# Semana 4: Normalización e Integridad Referencial
## Notebook Conceptual (NB1) – Desnormalización vs Normalización con pandas

### Propósito de la sesión
Aplicar las **formas normales** (1FN, 2FN, 3FN) para eliminar redundancias y garantizar consistencia en los datos. Partiremos de un DataFrame desnormalizado (con datos repetidos y redundantes) y lo transformaremos paso a paso, comparando el tamaño en memoria y la facilidad de consulta. Discutiremos los **trade-offs** entre rendimiento (consultas más rápidas en esquemas desnormalizados) y consistencia (integridad en esquemas normalizados).

### Objetivos de aprendizaje
*   Identificar **redundancias** y **anomalías** en una tabla desnormalizada.
*   Aplicar la **Primera Forma Normal (1FN)**: atributos atómicos y sin grupos repetitivos.
*   Aplicar la **Segunda Forma Normal (2FN)**: eliminar dependencias parciales.
*   Aplicar la **Tercera Forma Normal (3FN)**: eliminar dependencias transitivas.
*   Comparar el **tamaño en memoria** de las tablas normalizadas vs desnormalizadas.
*   Evaluar la **facilidad de consulta** (ej. actualizaciones, joins).
*   Discutir los **trade-offs** entre normalización y desnormalización.

## Configuración Inicial

Importamos las librerías necesarias: pandas para manipulación, numpy y matplotlib para análisis.

In [ ]:
# Importamos librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de pandas para mostrar más columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Creación de un DataFrame Desnormalizado

Simularemos una tabla de **pedidos** en una tienda online, con información redundante. Un cliente puede tener varios pedidos, y cada pedido puede tener varios productos (pero aquí lo simplificamos a un producto por fila).

In [ ]:
# Datos desnormalizados (redundantes)
data = {
    'id_pedido': [1001, 1001, 1002, 1003, 1003, 1004],
    'cliente_id': [1, 1, 2, 3, 3, 1],
    'cliente_nombre': ['Juan Pérez', 'Juan Pérez', 'María García', 'Carlos López', 'Carlos López', 'Juan Pérez'],
    'cliente_email': ['juan@email.com', 'juan@email.com', 'maria@email.com', 'carlos@email.com', 'carlos@email.com', 'juan@email.com'],
    'cliente_telefono': ['555-1234', '555-1234', '555-5678', '555-9012', '555-9012', '555-1234'],
    'cliente_ciudad': ['Madrid', 'Madrid', 'Barcelona', 'Valencia', 'Valencia', 'Madrid'],
    'producto_id': [101, 102, 101, 103, 104, 105],
    'producto_nombre': ['Laptop', 'Mouse', 'Laptop', 'Monitor', 'Teclado', 'Auriculares'],
    'producto_categoria': ['Electrónica', 'Electrónica', 'Electrónica', 'Electrónica', 'Electrónica', 'Electrónica'],
    'precio_unitario': [1200.00, 25.50, 1200.00, 300.00, 45.00, 80.00],
    'cantidad': [1, 2, 1, 1, 1, 1],
    'fecha_pedido': ['2025-03-01', '2025-03-01', '2025-03-02', '2025-03-02', '2025-03-02', '2025-03-03']
}

df_denormalized = pd.DataFrame(data)

print("🔷 DataFrame desnormalizado:")
display(df_denormalized)

In [ ]:
# Calculamos el tamaño en memoria del DataFrame original
mem_original = df_denormalized.memory_usage(deep=True).sum()
print(f"Tamaño en memoria del DataFrame desnormalizado: {mem_original / 1024:.2f} KB")

### Anomalías identificadas

*   **Redundancia**: Los datos del cliente (nombre, email, teléfono, ciudad) se repiten en cada fila de pedido.
*   **Redundancia**: Los datos del producto (nombre, categoría, precio) también se repiten.
*   **Anomalía de actualización**: Si Juan Pérez cambia de teléfono, hay que actualizar múltiples filas.
*   **Anomalía de inserción**: No se puede añadir un nuevo producto sin un pedido.
*   **Anomalía de eliminación**: Si se elimina el único pedido de un producto, se pierde la información del producto.

---
## 2. Aplicación de la Primera Forma Normal (1FN)

**Regla de 1FN**: Los atributos deben ser atómicos (no repetir grupos). Nuestro DataFrame ya cumple porque cada columna tiene valores escalares y no hay listas. Pero a veces la 1FN implica separar grupos repetitivos. En nuestro caso, podríamos tener una tabla de pedidos y otra de líneas de pedido. Ya lo tenemos implícito (cada fila es una línea). Así que nuestro DataFrame está en 1FN.

**Pero**, para practicar, supongamos que originalmente teníamos una columna 'productos' con una lista. Lo transformamos a filas separadas. (Ya lo hicimos).

In [ ]:
# Verificamos atomicidad
print("Tipos de datos:")
print(df_denormalized.dtypes)
print("\n✅ El DataFrame está en 1FN (todos los valores son atómicos).")

---
## 3. Aplicación de la Segunda Forma Normal (2FN)

**Regla de 2FN**: Estar en 1FN y todo atributo no clave debe depender **completamente** de la clave primaria (no de parte de ella).

En nuestro DataFrame, la clave primaria podría ser compuesta: `(id_pedido, producto_id)`. Analicemos dependencias:
*   `cantidad` depende de la clave completa (qué producto y en qué pedido). ✅
*   `precio_unitario` depende solo de `producto_id` (el precio de un producto no varía por pedido). ❌ Esto viola 2FN.
*   Los datos del cliente (`cliente_nombre`, `cliente_email`, ...) dependen solo de `cliente_id`, no de la clave completa. ❌
*   Los datos del producto (`producto_nombre`, `producto_categoria`) dependen solo de `producto_id`. ❌

Para 2FN, separamos en tablas:
*   **Clientes**
*   **Productos**
*   **Pedidos** (cabecera)
*   **DetallePedido** (líneas)

In [ ]:
# Creamos tabla de clientes (sin duplicados)
df_cliente = df_denormalized[['cliente_id', 'cliente_nombre', 'cliente_email', 'cliente_telefono', 'cliente_ciudad']].drop_duplicates().reset_index(drop=True)
print("🔷 Tabla Cliente (2FN):")
display(df_cliente)

# Creamos tabla de productos
df_producto = df_denormalized[['producto_id', 'producto_nombre', 'producto_categoria', 'precio_unitario']].drop_duplicates().reset_index(drop=True)
print("\n🔷 Tabla Producto (2FN):")
display(df_producto)

# Creamos tabla de pedidos (cabecera)
df_pedido = df_denormalized[['id_pedido', 'cliente_id', 'fecha_pedido']].drop_duplicates().reset_index(drop=True)
print("\n🔷 Tabla Pedido (2FN):")
display(df_pedido)

# Creamos tabla de detalle de pedido (líneas)
df_detalle = df_denormalized[['id_pedido', 'producto_id', 'cantidad']].copy()
# Añadimos precio_unitario del momento (lo tomamos de la tabla producto, pero en un sistema real se guardaría en detalle)
# Para mantener el histórico, deberíamos guardar el precio en detalle. Aquí lo añadimos.
df_detalle = df_detalle.merge(df_producto[['producto_id', 'precio_unitario']], on='producto_id', how='left')
print("\n🔷 Tabla DetallePedido (2FN):")
display(df_detalle)

In [ ]:
# Calculamos memoria total de las tablas normalizadas (2FN)
mem_cliente = df_cliente.memory_usage(deep=True).sum()
mem_producto = df_producto.memory_usage(deep=True).sum()
mem_pedido = df_pedido.memory_usage(deep=True).sum()
mem_detalle = df_detalle.memory_usage(deep=True).sum()
mem_total_2fn = mem_cliente + mem_producto + mem_pedido + mem_detalle

print(f"Memoria original desnormalizada: {mem_original / 1024:.2f} KB")
print(f"Memoria total en 2FN: {mem_total_2fn / 1024:.2f} KB")
print(f"Reducción: {100 * (mem_original - mem_total_2fn) / mem_original:.2f}%")

---
## 4. Aplicación de la Tercera Forma Normal (3FN)

**Regla de 3FN**: Estar en 2FN y ningún atributo no clave debe depender transitivamente de otro atributo no clave.

En nuestras tablas actuales:
*   En **Cliente**, todos los atributos dependen directamente de `cliente_id`. No hay dependencias transitivas. ✅
*   En **Producto**, todos los atributos dependen de `producto_id`. ✅
*   En **Pedido**, `fecha_pedido` depende de `id_pedido`; `cliente_id` es FK. ✅
*   En **DetallePedido**, `cantidad` y `precio_unitario` dependen de la clave compuesta `(id_pedido, producto_id)`. El precio podría depender del producto, pero en un sistema real, el precio en el detalle es el precio en el momento de la venta, por lo que es correcto. Sin embargo, si `precio_unitario` se obtuviera de la tabla producto, habría que mantenerlo en detalle para histórico. Nuestra tabla ya lo incluye. ✅

Nuestras tablas ya están en 3FN. Veamos un ejemplo de posible violación de 3FN: si en Cliente tuviéramos `ciudad_nombre` y `ciudad_codigo_postal` y esta última dependiera de la ciudad, habría que separar una tabla Ciudad.

In [ ]:
# Ejemplo de violación de 3FN (para ilustrar)
df_cliente_con_ciudad = df_cliente.copy()
df_cliente_con_ciudad['ciudad_cp'] = ['28001', '08001', '46001']  # Códigos postales ficticios
print("Tabla Cliente con posible dependencia transitiva (ciudad -> cp):")
display(df_cliente_con_ciudad)

# Para 3FN, separaríamos Ciudad
df_ciudad = df_cliente_con_ciudad[['cliente_ciudad', 'ciudad_cp']].drop_duplicates().reset_index(drop=True)
df_cliente_3fn = df_cliente_con_ciudad[['cliente_id', 'cliente_nombre', 'cliente_email', 'cliente_telefono', 'cliente_ciudad']]
print("\nTabla Ciudad (3FN):")
display(df_ciudad)
print("\nTabla Cliente (3FN, sin cp):")
display(df_cliente_3fn)

---
## 5. Comparación: Tamaño en Memoria y Facilidad de Consulta

### 5.1. Tamaño en memoria

Ya calculamos la reducción al pasar a 2FN. La 3FN apenas añade una tabla extra, pero en nuestro ejemplo no mejora significativamente el tamaño.

### 5.2. Facilidad de consulta

Veamos cómo cambian las consultas típicas:

In [ ]:
# Consulta en tabla desnormalizada: obtener pedidos de un cliente con detalles de producto
cliente_buscar = 1
consulta_denorm = df_denormalized[df_denormalized['cliente_id'] == cliente_buscar]
print("🔷 Consulta en tabla desnormalizada:")
display(consulta_denorm[['id_pedido', 'producto_nombre', 'cantidad', 'precio_unitario']])

# Consulta en tablas normalizadas (requiere join)
consulta_norm = (df_pedido[df_pedido['cliente_id'] == cliente_buscar]
                 .merge(df_detalle, on='id_pedido')
                 .merge(df_producto, on='producto_id')
                 [['id_pedido', 'producto_nombre', 'cantidad', 'precio_unitario']])
print("\n🔶 Consulta en tablas normalizadas (con joins):")
display(consulta_norm)

### 5.3. Actualización de datos

**Ventaja de la normalización:** Actualizar el teléfono de un cliente es sencillo y sin riesgo de inconsistencias.

In [ ]:
# Actualización en tabla normalizada (una sola fila)
print("Tabla Cliente antes:")
display(df_cliente[df_cliente['cliente_id'] == 1])

# Simulamos actualización (en pandas no persistimos)
df_cliente.loc[df_cliente['cliente_id'] == 1, 'cliente_telefono'] = '555-9999'
print("\nTabla Cliente después (una fila actualizada):")
display(df_cliente[df_cliente['cliente_id'] == 1])

# En desnormalizada, habría que actualizar múltiples filas
print("\nEn tabla desnormalizada, habría que actualizar todas estas filas:")
display(df_denormalized[df_denormalized['cliente_id'] == 1])

---
## 6. Discusión: Trade-offs entre Normalización y Desnormalización

| Aspecto | Normalizado | Desnormalizado |
|---------|-------------|----------------|
| **Redundancia** | Baja (almacena cada dato una vez) | Alta (datos repetidos) |
| **Consistencia** | Alta (actualización única) | Riesgo de inconsistencias |
| **Tamaño** | Menor (en teoría, pero más tablas) | Mayor (por redundancia) |
| **Velocidad de consulta** | Puede requerir JOINs (lentos si no se indexan) | Consultas simples y rápidas (todo en una tabla) |
| **Mantenimiento** | Más complejo (múltiples tablas) | Sencillo (una tabla) |
| **Escrituras (INSERT/UPDATE)** | Más rápidas por tabla (si están indexadas) | Lento por posibles bloqueos y mayor volumen |
| **Casos de uso** | Sistemas OLTP (transaccionales) | Data Warehouses (OLAP), reporting, caching |

**Conclusión:** No hay una solución única. Se debe equilibrar según las necesidades: normalizar para transacciones y consistencia; desnormalizar para análisis y rendimiento de lecturas.

---
## Ejercicios para el Estudiante

1.  **Detectar más violaciones:** En la tabla Cliente, ¿podría haber dependencia transitiva si añadimos `ciudad` y `pais`? ¿Cómo normalizarías?

2.  **Ampliar el dataset:** Añade más pedidos y productos al DataFrame original. Calcula nuevamente la reducción de memoria tras la normalización.

3.  **Simular actualización masiva:** En la tabla desnormalizada, actualiza el email de un cliente que aparece en 10 pedidos. Compara el número de filas afectadas con la tabla normalizada.

4.  **Desnormalización controlada:** A partir de las tablas normalizadas, crea una vista desnormalizada que combine todos los datos (como el DataFrame original). Mide su tamaño y reflexiona sobre su utilidad.

5.  **Reflexión:** ¿En qué situaciones preferirías un esquema desnormalizado a pesar de la redundancia? Pon ejemplos del mundo real.

---
## Conclusiones de la Sesión

*   Partimos de un DataFrame **desnormalizado** con claras redundancias y anomalías.
*   Aplicamos la **1FN** (verificando atomicidad).
*   Aplicamos la **2FN** separando clientes, productos, pedidos y detalles, eliminando dependencias parciales.
*   Aplicamos la **3FN** eliminando dependencias transitivas (ejemplo de ciudad y código postal).
*   Comparamos el **tamaño en memoria**: las tablas normalizadas ocuparon menos espacio gracias a la eliminación de redundancias.
*   Evaluamos la **facilidad de consulta**: las consultas normalizadas requieren joins, mientras que la desnormalizada es más simple pero menos mantenible.
*   Discutimos los **trade-offs** entre normalización (consistencia, menor redundancia) y desnormalización (rendimiento de lectura, simplicidad).

¡Ahora comprendes los principios de normalización y cómo aplicarlos en la práctica!